In [ ]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

In [ ]:
import subprocess
import time
import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


In [ ]:
!ollama pull llama3.2

In [ ]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-23T06:58:11.491007159Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":13472684995,"load_duration":9258930198,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":3608920000,"eval_count":3,"eval_duration":601291000}

In [ ]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))

.env created. Key length: 53


In [ ]:
!ls -a

.  ..  .env


In [ ]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from google import genai

print("google-genai imported OK")

google-genai imported OK


In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests

# Common interface
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Writing llm_client.py


In [ ]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

Writing main.py


In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests

# Common interface
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Overwriting llm_client.py


In [ ]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

Overwriting main.py


In [ ]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient

# Fake provider
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")

        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()

Writing test_client.py


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.80s ===============================


In [ ]:
import subprocess
import time
import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

print("Ollama server started!")

Ollama server started!


In [ ]:
!curl http://localhost:11434

Ollama is running

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Explain Artificial Intelligence in two sentences.",
        "stream": False
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["response"])

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, using algorithms and data. AI systems can be trained on large amounts of data to recognize patterns, make predictions, and take actions autonomously, with the goal of improving efficiency, accuracy, and productivity in various industries and applications.


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.97s ===============================


In [ ]:
!ls -l

total 20
-rw-r--r-- 1 root root 1270 Nov 23 07:04 llm_client.py
-rw-r--r-- 1 root root  701 Nov 23 07:05 main.py
drwxr-xr-x 2 root root 4096 Nov 23 07:05 __pycache__
-rw-r--r-- 1 root root   43 Nov 23 07:03 requirements.txt
-rw-r--r-- 1 root root  523 Nov 23 07:05 test_client.py


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 2.40s ===============================


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

key = os.getenv("GEMINI_API_KEY")

if not key:
    print("API key not found")
else:
    client = genai.Client(api_key=key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Explain Artificial Intelligence in two sentences."
    )
    print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

key = os.getenv("GEMINI_API_KEY")

if key:
    client = genai.Client(api_key=key)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Explain Artificial Intelligence in two sentences."
    )

    print(response.text)
else:
    print("GEMINI_API_KEY is missing.")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

try:
    load_dotenv(override=True)
    key = os.getenv("GEMINI_API_KEY")

    if not key:
        print("Error: Gemini API key is missing. Check Colab Secrets.")
    else:
        client = genai.Client(api_key=key)

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents="Explain Artificial Intelligence in two sentences."
        )

        print("Gemini response:")
        print(response.text)

except Exception as e:
    print("Error:", e)

Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

try:
    load_dotenv(override=True)
    key = os.getenv("GEMINI_API_KEY")

    if not key:
        print("API key missing.")
    else:
        client = genai.Client(api_key=key)

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="Explain Artificial Intelligence in two sentences."
        )

        print(response.text)

except Exception as e:
    print("Error:", e)

Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.54s ===============================


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.52s ===============================


In [ ]:
!ls -la

total 36
drwxr-xr-x 4 root root 4096 Nov 23 07:05 .
drwxr-xr-x 1 root root 4096 Nov 23 06:55 ..
-rw-r--r-- 1 root root   71 Nov 23 07:02 .env
-rw-r--r-- 1 root root 1270 Nov 23 07:04 llm_client.py
-rw-r--r-- 1 root root  701 Nov 23 07:05 main.py
drwxr-xr-x 2 root root 4096 Nov 23 07:05 __pycache__
drwxr-xr-x 3 root root 4096 Nov 23 07:05 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 23 07:03 requirements.txt
-rw-r--r-- 1 root root  523 Nov 23 07:05 test_client.py
